# TruthLens Audio -- v7: binary Real vs Deepfake, trained on diverse deepfake data

**Scope:** deepfake = a cloned/converted voice impersonating a real person. Two classes only
(`genuine`, `deepfake`). Generic text-to-speech data (WaveFake, adarshsingh, FoR) is NOT used.

**What changed vs v5/v6** (which only ever saw 3 speakers of phone-call-style audio and
failed on new speakers): much more varied real+deepfake audio, so the only thing all of it has
in common is real-vs-fake:
- **In-The-Wild** (real-world deepfakes of public figures + their real recordings), split **by
  speaker** so the test people are never heard in training.
- FakeAVCeleb is **held out entirely** as a cross-dataset test (new generator, new people).
- Every dataset contributes the **same number of real and fake clips**, so "which dataset is
  this?" can never predict the label (the shortcut behind most earlier failures).
- Phone-line distortion + noise augmentation, applied equally to real and fake clips.

**Recipe:** gentle fine-tune from v2's weights (lr 1e-5) -- the only recipe that has not collapsed.
The output layer is initialised from v2's `genuine` and `deepfake` rows.

**Pass mark (fixed in advance):** >= 80% of real clips AND >= 80% of fake clips correct, on data
the model never trained on. The final cell prints PASS/FAIL per test set, next to v5 for reference.
Nothing is integrated into the app unless this passes AND you approve.

Run top to bottom. You will be asked to upload `audio_eval_dataset.zip` once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

PROJECT_DIR = '/content/drive/MyDrive/TruthLens_Audio'
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

LABELS = ['genuine', 'deepfake']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}

MANIFEST_DIR = f'{PROJECT_DIR}/manifests_v2'       # v2's manifest (ASVspoof + FakeAVCeleb ...)
MANIFEST_DIR_V4 = f'{PROJECT_DIR}/manifests_v4'    # has the DEEP-VOICE chunk rows
export_dir = f'{PROJECT_DIR}/export_v2'            # starting weights
CKPT_DIR_V5 = f'{PROJECT_DIR}/checkpoints_v5'      # only for the v5 reference column

RUN_NAME_V7 = 'v7'
CKPT_DIR_V7 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V7}'
LOG_DIR_V7 = f'{PROJECT_DIR}/logs_{RUN_NAME_V7}'
MANIFEST_DIR_V7 = f'{PROJECT_DIR}/manifests_{RUN_NAME_V7}'
for d in [CKPT_DIR_V7, LOG_DIR_V7, MANIFEST_DIR_V7]:
    os.makedirs(d, exist_ok=True)

# Drive space check -- read-only, deletes nothing. A full Drive is what silently broke
# checkpoint saving before, so look at it up front.
free_gb = shutil.disk_usage(PROJECT_DIR).free / 1e9
print(f'Drive free space: {free_gb:.2f} GB')
big = []
for root, _, files in os.walk(PROJECT_DIR):
    for f in files:
        p = os.path.join(root, f)
        try:
            sz = os.path.getsize(p)
        except OSError:
            continue
        if sz > 300e6:
            big.append((sz, p))
for sz, p in sorted(big, reverse=True):
    print(f'  {sz/1e9:.2f} GB  {p}')
if free_gb < 2.5:
    print('\nWARNING: under 2.5 GB free. v7 saves a ~1.1 GB latest.pt each epoch plus best.pt '
          '(~0.4 GB), so saves may fail. The old checkpoints_v4/v5/v6 latest.pt files listed above '
          'only matter for resuming runs that are already finished -- freeing space is YOUR '
          'decision, nothing here deletes anything. The save check below will print '
          'drive_saved=False if a save fails.')

print('Setup complete.')


In [ ]:
!pip install -q transformers accelerate soundfile librosa scikit-learn kaggle pandas tqdm av


In [ ]:
import os, shutil
from pathlib import Path

# Safety net only -- restores below should already provide everything.
kaggle_cred_drive = Path(f'{PROJECT_DIR}/kaggle.json')

if not kaggle_cred_drive.exists():
    print('No cached Kaggle credential found in Drive -- please upload one now.')
    print('(Kaggle -> Settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, 'Expected a file to be uploaded'
    uploaded_name = next(iter(uploaded))
    shutil.copy(uploaded_name, kaggle_cred_drive)
    print('Cached to Drive for next time.')

raw = kaggle_cred_drive.read_text().strip()

if raw.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = raw
    legacy_path = Path(os.path.expanduser('~/.kaggle/kaggle.json'))
    if legacy_path.exists():
        legacy_path.unlink()
    print('Detected new-style KGAT_ token -> set KAGGLE_API_TOKEN env var.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(kaggle_cred_drive, os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Detected classic kaggle.json -> copied to ~/.kaggle/kaggle.json.')

print('Kaggle credentials ready.')


## Restore audio from the existing Drive backups (v2 data + DEEP-VOICE chunks)

In [ ]:
import subprocess

backup_path = f'{PROJECT_DIR}/data_backup.tar.gz'
assert os.path.exists(backup_path), f'No backup found at {backup_path} -- nothing to restore.'

subprocess.run(['cp', backup_path, '/content/data_backup.tar.gz'], check=True)
subprocess.run(['tar', '-xzf', '/content/data_backup.tar.gz', '-C', '/content'], check=True)
print('Restored v2 raw audio data (ASVspoof + FakeAVCeleb + WaveFake) from Drive backup.')


In [ ]:
import subprocess

V4_DATA_BACKUP = f'{PROJECT_DIR}/v4_data_backup.tar.gz'
assert os.path.exists(V4_DATA_BACKUP), f'{V4_DATA_BACKUP} not found'
subprocess.run(['cp', V4_DATA_BACKUP, '/content/v4_data_backup.tar.gz'], check=True)
try:
    # only the pre-chunked DEEP-VOICE clips are needed here (not the generic-TTS adarshsingh set)
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content',
                    'deepvoice_chunks'], check=True)
except subprocess.CalledProcessError:
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content'], check=True)

# The backup extracts one level above DATA_DIR; the manifest paths expect DATA_DIR/deepvoice_chunks.
link = f'{DATA_DIR}/deepvoice_chunks'
if os.path.exists('/content/deepvoice_chunks') and not os.path.exists(link):
    os.symlink('/content/deepvoice_chunks', link)
assert os.path.isdir(link) and os.listdir(link), 'deepvoice_chunks missing after restore'
print('DEEP-VOICE chunks ready:', len(os.listdir(link)), 'files')


## Helpers, model class, augmentation

In [ ]:
import random
import numpy as np
import pandas as pd
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
import soundfile as sf
import librosa
from pathlib import Path
from tqdm import tqdm

TARGET_SR = 16000
WINDOW_SAMPLES = TARGET_SR * 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU -- training will be far too slow. Runtime -> Change runtime type -> GPU.')

def load_mono_16k(path):
    # soundfile can't open some containers (notably .m4a) -- fall back to librosa (ffmpeg).
    try:
        audio, sr = sf.read(path, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
        return audio.astype(np.float32)
    except Exception:
        audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
        return audio.astype(np.float32)

def rms_normalize(audio, target_dbfs=-20.0):
    rms = np.sqrt(np.mean(audio ** 2)) + 1e-9
    return audio * (10 ** (target_dbfs / 20) / rms)

def fixed_window(audio, length, train=False):
    if len(audio) >= length:
        start = np.random.randint(0, len(audio) - length + 1) if train else (len(audio) - length) // 2
        return audio[start:start + length]
    pad = length - len(audio)
    left = pad // 2
    return np.pad(audio, (left, pad - left), mode='reflect')

def _match_len(x, n):
    return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))

def phone_quality(audio):
    # band-limit like a phone line: 8 kHz sampling (~4 kHz bandwidth), then back to 16 kHz
    low = librosa.resample(audio, orig_sr=TARGET_SR, target_sr=8000)
    return _match_len(librosa.resample(low, orig_sr=8000, target_sr=TARGET_SR), len(audio)).astype(np.float32)

def add_noise(audio):
    snr_db = random.uniform(15, 35)
    noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
    return (audio + np.random.randn(len(audio)) * np.sqrt(noise_power)).astype(np.float32)

def augment_audio(audio):
    # Same probabilities for EVERY clip, whatever its label. If only one class were degraded the
    # model could learn "phone-quality = fake" (or real) instead of the actual difference.
    if random.random() < 0.25:
        audio = phone_quality(audio)
    if random.random() < 0.25:
        audio = add_noise(audio)
    return rms_normalize(audio)

class AudioDatasetV7(Dataset):
    def __init__(self, df, train, augment=False):
        self.rows = df.reset_index(drop=True)
        self.train = train
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        try:
            audio = load_mono_16k(row['path'])
            audio = rms_normalize(audio)
            audio = fixed_window(audio, WINDOW_SAMPLES, self.train)
            if self.augment:
                audio = augment_audio(audio)
        except Exception:
            audio = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        return torch.from_numpy(audio.astype(np.float32)), LABEL_TO_IDX[row['label']]

class AudioClassifier(nn.Module):
    def __init__(self, num_classes=2, freeze_feature_extractor=True):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        if freeze_feature_extractor:
            self.backbone.feature_extractor._freeze_parameters()
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))

    def forward(self, waveform):
        out = self.backbone(waveform).last_hidden_state
        return self.head(out.mean(dim=1))

def collate(batch):
    return torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch])


## Upload the bank-fraud voice-changing set (used as one training source + one test set)

In [ ]:
from google.colab import files
import zipfile

print('Upload the zipped "audio" folder (voice_changing dataset).')
uploaded = files.upload()
zip_name = next(iter(uploaded))

eval_dir = '/content/real_eval_audio'
os.makedirs(eval_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(eval_dir)
print('Extracted to', eval_dir)


In [ ]:
import glob

eval_files = (
    glob.glob(f'{eval_dir}/**/*.mp3', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.m4a', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.wav', recursive=True)
)
print(f'Found {len(eval_files)} real-world eval files.')
assert eval_files, 'No audio files found in the extracted zip -- check its folder structure.'


In [ ]:
from pathlib import Path

HELD_OUT_SPEAKER = ('USA', 'male')  # never used in training -- pure held-out eval target

realistic_finetune_rows = []
realistic_heldout_rows = []
for path in eval_files:
    parts = Path(path).parts
    country, gender = parts[-4], parts[-3]
    stem = Path(path).stem
    label = 'genuine' if stem == 'original' else 'deepfake'
    row = {'path': path, 'label': label, 'source': 'realistic_voice_changing',
           'speaker_id': f'{country}_{gender}', 'split_hint': None}
    if (country, gender) == HELD_OUT_SPEAKER:
        realistic_heldout_rows.append(row)
    else:
        realistic_finetune_rows.append(row)

print(f'Fine-tune rows (3 speakers): {len(realistic_finetune_rows)}')
print(f'Held-out eval rows (speaker {HELD_OUT_SPEAKER}): {len(realistic_heldout_rows)}')
assert realistic_finetune_rows and realistic_heldout_rows, 'Expected data on both sides of the speaker split.'


## In-The-Wild: download, split by speaker, keep a compact subset

Real-world audio deepfakes of public figures plus real recordings of the same people. Split
**by speaker** (train / val / test people never overlap). A compact FLAC subset is backed up to
Drive as ONE archive, so a new session restores it instead of re-downloading ~9 GB.

In [ ]:
import subprocess, zipfile

ITW_DIR = f'{DATA_DIR}/itw_subset'
ITW_BACKUP = f'{PROJECT_DIR}/itw_subset_backup.tar.gz'
ITW_MANIFEST = f'{MANIFEST_DIR_V7}/itw_manifest.csv'
ITW_MIRRORS = ['abdallamohamed312/in-the-wild-audio-deepfake',   # real/ + fake/ folders + meta.csv
               'abdallamohamed312/in-the-wild-dataset']          # backup copy (original release layout)
ITW_MIN_FILES = 30000     # the full dataset has 31,779 clips; fewer than this means a partial download
NESTED_MIN_BYTES = 1e9    # a zip bigger than this inside the download is treated as the real data archive

# caps per class per split (total ~8k clips)
ITW_CAP = {'train': 2500, 'val': 400, 'test': 800}
MAX_CLIP_SECONDS = 12

def kaggle_credentials_status():
    has_env = bool(os.environ.get('KAGGLE_API_TOKEN'))
    has_file = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
    print(f'Kaggle credentials in this session -> env token: {has_env}, kaggle.json: {has_file}')
    if not (has_env or has_file):
        print('  NO Kaggle credentials found. Run the "Kaggle credentials" cell near the top first.')

def list_dir(raw_dir):
    for name in sorted(os.listdir(raw_dir)):
        p = os.path.join(raw_dir, name)
        size = os.path.getsize(p) if os.path.isfile(p) else 0
        print(f'    {name}  {"(dir)" if os.path.isdir(p) else f"{size / 1e9:.2f} GB"}')

def extract_nested_archives(raw_dir):
    # This Kaggle mirror ships the data as a zip INSIDE the downloaded archive (a file literally named
    # 'download'), so --unzip leaves a big zip behind. Unpack it (no re-download needed).
    if not os.path.isdir(raw_dir):
        return
    for _ in range(3):
        if len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True)) >= ITW_MIN_FILES:
            return
        found = False
        for p in glob.glob(f'{raw_dir}/**/*', recursive=True):
            if os.path.isfile(p) and os.path.getsize(p) > NESTED_MIN_BYTES and zipfile.is_zipfile(p):
                print(f'  Unpacking nested archive {p} ({os.path.getsize(p) / 1e9:.2f} GB) ...')
                subprocess.run(['unzip', '-q', '-o', p, '-d', f'{raw_dir}/extracted'])
                os.remove(p)   # free the disk space; the extracted copy is what we use
                found = True
        if not found:
            return

def download_itw(raw_dir):
    kaggle_credentials_status()
    for ref in ITW_MIRRORS:
        shutil.rmtree(raw_dir, ignore_errors=True)   # never build on top of a partial download
        os.makedirs(raw_dir, exist_ok=True)
        print(f'Downloading {ref} (~9 GB, roughly 5-15 min, no progress bar) ...')
        r = subprocess.run(['kaggle', 'datasets', 'download', '-d', ref, '-p', raw_dir, '--unzip'],
                           capture_output=True, text=True)
        if r.returncode == 0:
            extract_nested_archives(raw_dir)
        n = len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True))
        if r.returncode == 0 and n >= ITW_MIN_FILES:
            print(f'  OK: {n} wav files')
            return ref
        print(f'  FAILED (exit code {r.returncode}, {n} wav files found). Kaggle said:')
        print('  ' + ((r.stderr or '') + (r.stdout or ''))[-800:].replace('\n', '\n  '))
        print('  Files left in the download folder:')
        list_dir(raw_dir)
    raise RuntimeError('Could not download In-The-Wild from any mirror -- read the Kaggle messages above.')

def meta_label_to_class(x):
    x = str(x).lower()
    if x in ('bona-fide', 'bonafide', 'bona_fide', 'real', 'genuine'):
        return 'genuine'
    if x in ('spoof', 'fake', 'deepfake'):
        return 'deepfake'
    return None

def restore_itw():
    subprocess.run(['cp', ITW_BACKUP, '/content/itw_subset_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/itw_subset_backup.tar.gz', '-C', DATA_DIR], check=True)
    return pd.read_csv(ITW_MANIFEST)

def build_itw():
    raw_dir = f'{DATA_DIR}/itw_raw'
    extract_nested_archives(raw_dir)   # salvage a download already on disk instead of fetching 9 GB again
    if len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True)) < ITW_MIN_FILES:
        download_itw(raw_dir)

    all_wavs = glob.glob(f'{raw_dir}/**/*.wav', recursive=True)
    meta_path = glob.glob(f'{raw_dir}/**/meta.csv', recursive=True)
    assert meta_path, 'meta.csv (speaker info) not found'
    meta = pd.read_csv(meta_path[0])
    print('meta.csv columns:', meta.columns.tolist())
    print(meta.head(3).to_string())
    file_col = next(c for c in meta.columns if 'file' in c.lower())
    spk_col = next(c for c in meta.columns if 'speaker' in c.lower())
    label_col = next((c for c in meta.columns if 'label' in c.lower()), None)
    meta['_name'] = meta[file_col].astype(str).map(lambda x: Path(x).name if x.lower().endswith('.wav') else x + '.wav')
    speaker_of = dict(zip(meta['_name'], meta[spk_col].astype(str)))

    # label: from the real/ fake/ folders if present, otherwise from meta.csv (original release layout)
    folder_label = {}
    for p in all_wavs:
        parent = Path(p).parent.name.lower()
        if parent in ('real', 'fake'):
            folder_label[p] = 'genuine' if parent == 'real' else 'deepfake'
    if len(folder_label) < 0.9 * len(all_wavs):
        assert label_col is not None, 'No real/fake folders and no label column in meta.csv'
        name_to_label = dict(zip(meta['_name'], meta[label_col].map(meta_label_to_class)))
        folder_label = {p: name_to_label.get(Path(p).name) for p in all_wavs}
    rows = []
    for p, label in folder_label.items():
        spk = speaker_of.get(Path(p).name)
        if label is not None and spk is not None:
            rows.append({'path': p, 'label': label, 'speaker_id': spk})
    df = pd.DataFrame(rows)
    print(f'{len(df)} clips matched to a speaker and a label; classes:', df['label'].value_counts().to_dict())
    assert df['label'].nunique() == 2, 'Need both real and fake clips'
    if label_col is not None:  # sanity: our label vs meta's label agree
        name_to_meta = dict(zip(meta['_name'], meta[label_col].astype(str)))
        df['_meta_label'] = df['path'].map(lambda p: name_to_meta.get(Path(p).name))
        print(pd.crosstab(df['label'], df['_meta_label']))

    # split BY SPEAKER; both classes of a person stay in the same split
    speakers = sorted(df['speaker_id'].unique())
    rs = np.random.RandomState(42)
    rs.shuffle(speakers)
    n_test, n_val = max(1, round(0.2 * len(speakers))), max(1, round(0.1 * len(speakers)))
    spk_split = {}
    for i, s in enumerate(speakers):
        spk_split[s] = 'test' if i < n_test else ('val' if i < n_test + n_val else 'train')
    df['split'] = df['speaker_id'].map(spk_split)
    print(f'{len(speakers)} speakers -> train {sum(v == "train" for v in spk_split.values())}, '
          f'val {n_val}, test {n_test}')

    # subset: equal real/fake per split, capped
    parts = []
    for split, cap in ITW_CAP.items():
        d = df[df['split'] == split]
        n = min((d['label'] == 'genuine').sum(), (d['label'] == 'deepfake').sum(), cap)
        for label in LABELS:
            parts.append(d[d['label'] == label].sample(n, random_state=42))
    sub = pd.concat(parts).reset_index(drop=True)

    # convert to 16 kHz mono FLAC (compact + one uniform container for every clip)
    new_paths = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc='converting to flac'):
        out_dir = f'{ITW_DIR}/{r["split"]}/{r["label"]}'
        os.makedirs(out_dir, exist_ok=True)
        out = f'{out_dir}/{i}.flac'
        if not os.path.exists(out):
            audio = load_mono_16k(r['path'])[:MAX_CLIP_SECONDS * TARGET_SR]
            sf.write(out, audio, TARGET_SR, format='FLAC')
        new_paths.append(out)
    sub['path'] = new_paths
    sub['source'] = 'inthewild'
    sub = sub[['path', 'label', 'source', 'speaker_id', 'split']]
    sub.to_csv(ITW_MANIFEST, index=False)

    subprocess.run(['tar', '-czf', '/content/itw_subset_backup.tar.gz', '-C', DATA_DIR, 'itw_subset'], check=True)
    size = os.path.getsize('/content/itw_subset_backup.tar.gz')
    free = shutil.disk_usage(PROJECT_DIR).free
    if size * 1.05 < free:
        shutil.copy2('/content/itw_subset_backup.tar.gz', ITW_BACKUP)
        ok = os.path.exists(ITW_BACKUP) and os.path.getsize(ITW_BACKUP) == size
        print(f'ITW backup to Drive: {size / 1e9:.2f} GB, saved={ok}')
    else:
        print(f'WARNING: Drive too full for the {size / 1e9:.2f} GB ITW backup ({free / 1e9:.2f} GB free) -- '
              'skipped. Next session will re-download In-The-Wild.')
    shutil.rmtree(raw_dir, ignore_errors=True)
    return sub

if os.path.exists(ITW_BACKUP) and os.path.exists(ITW_MANIFEST):
    print('Restoring In-The-Wild subset from Drive backup...')
    itw_df = restore_itw()
else:
    itw_df = build_itw()

print(itw_df.groupby(['split', 'label']).size().unstack())
print('speakers per split:', itw_df.groupby('split')['speaker_id'].nunique().to_dict())
train_spk = set(itw_df[itw_df.split == 'train'].speaker_id)
test_spk = set(itw_df[itw_df.split == 'test'].speaker_id)
assert not (train_spk & test_spk), 'speaker leak between train and test!'


## Assemble train / val / test

Rules: (1) only `genuine` and `deepfake` rows; (2) **every dataset family contributes the same
number of real and fake clips** (families with only one class are dropped, so the dataset can't
predict the label); (3) FakeAVCeleb is kept out of training as a cross-dataset test.

In [ ]:
CAP_TRAIN, CAP_VAL, CAP_TEST = 2500, 400, 800
REALISTIC_TARGET_PER_CLASS = 300      # mild oversampling of the 3 training speakers (v6 showed heavy oversampling doesn't help)

def family_of(src):
    s = str(src).lower()
    for key in ['asvspoof', 'fakeavceleb', 'wavefake', 'deepvoice', 'inthewild', 'realistic']:
        if key in s:
            return key
    return s

def balance_per_family(df, cap, seed=42):
    parts = []
    for fam, g in df.groupby('family'):
        gr, gf = g[g['label'] == 'genuine'], g[g['label'] == 'deepfake']
        n = min(len(gr), len(gf), cap)
        if n == 0:
            print(f'  dropping family {fam!r}: only one class present ({len(gr)} genuine / {len(gf)} deepfake)')
            continue
        parts += [gr.sample(n, random_state=seed), gf.sample(n, random_state=seed)]
    return pd.concat(parts) if parts else g.iloc[0:0]

# 1. v2 manifest: ASVspoof / FakeAVCeleb ... keep genuine vs deepfake only
df_v2 = pd.read_csv(f'{MANIFEST_DIR}/phase1_manifest.csv')
print('v2 manifest columns:', df_v2.columns.tolist())
print(df_v2.groupby(['source', 'label']).size())
df_v2 = df_v2[df_v2['label'].isin(LABELS)].copy()
df_v2['family'] = df_v2['source'].map(family_of)
fav_mask = df_v2['family'] == 'fakeavceleb'
assert fav_mask.any(), 'No FakeAVCeleb rows found -- check the source names printed above'
df_fav = df_v2[fav_mask]
df_v2 = df_v2[~fav_mask]

v2_train = balance_per_family(df_v2[df_v2['split'] == 'train'], CAP_TRAIN)
v2_val = balance_per_family(df_v2[df_v2['split'] == 'val'], CAP_VAL)
v2_test = balance_per_family(df_v2[df_v2['split'] == 'test'], CAP_TEST)
fav_test = balance_per_family(df_fav, CAP_TEST)

# 2. DEEP-VOICE (RVC voice conversion of public figures) -- training only
df_v4 = pd.read_csv(f'{MANIFEST_DIR_V4}/v4_manifest.csv')
dv = df_v4[df_v4['source'].isin(['deepvoice_real', 'deepvoice_fake'])].copy()
assert len(dv) > 0, 'No DEEP-VOICE rows in the v4 manifest'
dv['family'] = 'deepvoice'
dv_train = balance_per_family(dv, CAP_TRAIN)

# 3. In-The-Wild (speaker-split)
itw_df['family'] = 'inthewild'
itw_train = balance_per_family(itw_df[itw_df['split'] == 'train'], CAP_TRAIN)
itw_val = balance_per_family(itw_df[itw_df['split'] == 'val'], CAP_VAL)
itw_test = balance_per_family(itw_df[itw_df['split'] == 'test'], CAP_TEST)

# 4. bank-fraud voice-changing: 3 speakers train (mild oversample, equal real/fake), 1 speaker test
def oversample_to(rows, target):
    d = pd.DataFrame(rows)
    reps = int(np.ceil(target / len(d)))
    return pd.concat([d] * reps, ignore_index=True).sample(target, random_state=42)

realistic_train = pd.concat([
    oversample_to([r for r in realistic_finetune_rows if r['label'] == 'genuine'], REALISTIC_TARGET_PER_CLASS),
    oversample_to([r for r in realistic_finetune_rows if r['label'] == 'deepfake'], REALISTIC_TARGET_PER_CLASS),
], ignore_index=True)
realistic_train['family'] = 'realistic'
realistic_test = pd.DataFrame(realistic_heldout_rows)
realistic_test['family'] = 'realistic'

cols = ['path', 'label', 'family']
train_df = pd.concat([d[cols] for d in (v2_train, dv_train, itw_train, realistic_train)], ignore_index=True)
val_df = pd.concat([d[cols] for d in (v2_val, itw_val)], ignore_index=True)

TEST_SETS = {
    'T1 In-The-Wild, unseen speakers': itw_test[cols],
    'T2 bank-fraud, unseen speaker (20 files)': realistic_test[cols],
    'T3 ASVspoof, in-domain': v2_test[cols],
    'T4 FakeAVCeleb, unseen dataset': fav_test[cols],
}

print('\nTRAIN composition (clips per family and class):')
print(train_df.groupby(['family', 'label']).size().unstack(fill_value=0))
print('\nVAL:', len(val_df), ' TESTS:', {k: len(v) for k, v in TEST_SETS.items()})
for name, d in TEST_SETS.items():
    assert not (set(d['path']) & set(train_df['path'])), f'test/train path overlap in {name}'
print('No train/test file overlap.')


In [ ]:
def preflight(df, name, per_family=25):
    # Load a sample from every family: catches unreadable files BEFORE hours of training
    # (the dataset silently swaps unreadable clips for silence, which is what hid the .m4a bug).
    bad, checked = 0, 0
    for fam, g in df.groupby('family'):
        for p in g['path'].sample(min(per_family, len(g)), random_state=0):
            checked += 1
            try:
                a = load_mono_16k(p)
                if len(a) < 1600 or np.abs(a).max() < 1e-4:
                    bad += 1
            except Exception:
                bad += 1
    print(f'{name}: {bad}/{checked} unreadable or silent')
    return bad, checked

tot_bad = tot = 0
for nm, d in [('train', train_df), ('val', val_df)] + list(TEST_SETS.items()):
    b, c = preflight(d, nm)
    tot_bad += b
    tot += c
assert tot_bad / max(tot, 1) < 0.02, 'Too many unreadable clips -- fix before training'
print('Preflight OK.')


## Train v7 (resumable, verified Drive saves)

Starts from v2's weights; the 2-class output layer is initialised from v2's `genuine` and
`deepfake` output rows. `best.pt` is saved from the first epoch on (validation now includes
unseen In-The-Wild speakers, so it is no longer just v2's own data).

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import csv

V7_BATCH_SIZE = 8
V7_GRAD_ACCUM = 4
V7_EPOCHS = 5
V7_LR = 1e-5
PATIENCE = 2

model = AudioClassifier(num_classes=2).to(device)
v2_sd = torch.load(f'{export_dir}/wav2vec2_deepfake_classifier.pt', map_location=device)
new_sd = model.state_dict()
for k, v in v2_sd.items():
    new_sd[k] = v[[0, 2]].clone() if k.startswith('head.3.') else v   # v2 rows: 0=genuine, 2=deepfake
model.load_state_dict(new_sd)
print('Initialised from v2 (backbone + hidden layer copied, output layer = v2 genuine/deepfake rows).')

train_loader = DataLoader(AudioDatasetV7(train_df, train=True, augment=True), batch_size=V7_BATCH_SIZE,
                          shuffle=True, collate_fn=collate, num_workers=2)
val_loader = DataLoader(AudioDatasetV7(val_df, train=False), batch_size=V7_BATCH_SIZE,
                        shuffle=False, collate_fn=collate, num_workers=2)

counts = train_df['label'].value_counts()
weights = torch.tensor([1.0 / counts.get(l, 1) for l in LABELS], dtype=torch.float32)
weights = (weights / weights.sum() * len(LABELS)).to(device)
print('Class weights:', dict(zip(LABELS, weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(model.parameters(), lr=V7_LR)
total_steps = (len(train_loader) // V7_GRAD_ACCUM) * V7_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)

def evaluate_val(loader):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for waveforms, labels in loader:
            logits = model(waveforms.to(device))
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            labels_all.extend(labels.numpy())
    return f1_score(labels_all, preds, average='macro')

LOCAL_CKPT_DIR = '/content/checkpoints_v7_local'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

def save_ckpt_verified(obj, filename):
    local_path = f'{LOCAL_CKPT_DIR}/{filename}'
    drive_path = f'{CKPT_DIR_V7}/{filename}'
    torch.save(obj, local_path)
    local_size = os.path.getsize(local_path)
    try:
        shutil.copy2(local_path, drive_path)
        drive_ok = os.path.exists(drive_path) and os.path.getsize(drive_path) == local_size
    except OSError as e:
        print(f'  [ckpt] Drive write error: {e}')
        drive_ok = False
    print(f'  [ckpt] {filename}: local={local_size/1e6:.1f}MB  drive_saved={drive_ok}')
    if not drive_ok:
        print(f'  [ckpt] WARNING: Drive copy failed/incomplete for {filename} -- local copy at {local_path} '
              '(download it from the Colab file browser before this session ends).')
    return drive_ok

ckpt_path = f'{CKPT_DIR_V7}/latest.pt'
log_path = f'{LOG_DIR_V7}/training_log.csv'
start_epoch, best_val_f1, epochs_no_improve = 0, -1.0, 0

if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    start_epoch = state['epoch'] + 1
    best_val_f1 = state.get('best_val_f1', -1.0)
    epochs_no_improve = state.get('epochs_no_improve', 0)
    print(f'Resuming v7 from epoch {start_epoch} (best val macro-F1 so far: {best_val_f1:.4f})')
else:
    print('No checkpoint found -- starting v7 from epoch 0.')

if not os.path.exists(log_path):
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_macro_f1'])

for epoch in range(start_epoch, V7_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (waveforms, labels) in enumerate(tqdm(train_loader, desc=f'v7 epoch {epoch}')):
        waveforms, labels = waveforms.to(device), labels.to(device)
        loss = criterion(model(waveforms), labels) / V7_GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * V7_GRAD_ACCUM
        if (step + 1) % V7_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_loss = running_loss / len(train_loader)
    val_f1 = evaluate_val(val_loader)
    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}')
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_loss, val_f1])

    if val_f1 > best_val_f1:
        best_val_f1, epochs_no_improve = val_f1, 0
        save_ckpt_verified(model.state_dict(), 'best.pt')
    else:
        epochs_no_improve += 1

    save_ckpt_verified({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch,
                        'best_val_f1': best_val_f1, 'epochs_no_improve': epochs_no_improve}, 'latest.pt')

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nv7 training done. Best val macro-F1: {best_val_f1:.4f}')
print('Files on Drive:', sorted(os.listdir(CKPT_DIR_V7)))


## Evaluate on data the model never trained on (v7 vs v5 as reference)

v5 is the previous best (3-class); it is collapsed to real-vs-deepfake the same way the app would
(anything not `genuine` counts as fake). AUC is threshold-free: it shows whether the model
separates real from fake at all, even if the 0.5 cut-off is in the wrong place.

In [ ]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def predict_probs(m, df, batch_size=16):
    m.eval()
    loader = DataLoader(AudioDatasetV7(df, train=False), batch_size=batch_size, shuffle=False,
                        collate_fn=collate, num_workers=2)
    out = []
    for waveforms, _ in tqdm(loader, desc='predicting', leave=False):
        out.append(F.softmax(m(waveforms.to(device)), dim=-1).cpu().numpy())
    return np.concatenate(out)

def binary_metrics(y_true, p_fake, pred_fake):
    y_true, pred_fake = np.asarray(y_true), np.asarray(pred_fake)
    real, fake = y_true == 0, y_true == 1
    return {
        'n': len(y_true),
        'real_ok': (~pred_fake[real]).mean() if real.any() else np.nan,
        'fake_ok': pred_fake[fake].mean() if fake.any() else np.nan,
        'acc': (pred_fake == fake).mean(),
        'auc': roc_auc_score(y_true, p_fake) if real.any() and fake.any() else np.nan,
    }

# v7: best checkpoint (fall back to latest if best is missing)
best_path = f'{CKPT_DIR_V7}/best.pt'
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
else:
    model.load_state_dict(torch.load(f'{CKPT_DIR_V7}/latest.pt', map_location=device)['model'])
model.eval()

# v5 reference (3-class: 0=genuine, 1=ai_generated, 2=deepfake)
v5_model = None
v5_path = f'{CKPT_DIR_V5}/latest.pt'
if os.path.exists(v5_path):
    v5_model = AudioClassifier(num_classes=3).to(device)
    v5_model.load_state_dict(torch.load(v5_path, map_location=device)['model'])
    v5_model.eval()

rows = []
for name, d in TEST_SETS.items():
    y = d['label'].map(LABEL_TO_IDX).values
    p7 = predict_probs(model, d)
    m7 = binary_metrics(y, p7[:, 1], p7.argmax(1) == 1)
    rows.append({'test set': name, 'model': 'v7', **m7})
    if v5_model is not None:
        p5 = predict_probs(v5_model, d)
        m5 = binary_metrics(y, 1 - p5[:, 0], p5.argmax(1) != 0)
        rows.append({'test set': name, 'model': 'v5', **m5})

res = pd.DataFrame(rows)
res['PASS (>=80% real and fake)'] = (res['real_ok'] >= 0.8) & (res['fake_ok'] >= 0.8)
pd.set_option('display.width', 200)
print(res.round(3).to_string(index=False))
res.to_csv(f'{PROJECT_DIR}/v7_vs_v5_results.csv', index=False)

v7_rows = res[res['model'] == 'v7']
print('\nv7 passes on', int(v7_rows['PASS (>=80% real and fake)'].sum()), 'of', len(v7_rows), 'test sets.')
print('Note: T2 has only 5 real clips -- one clip moves it by 20 points; treat it as a sanity check.')


## Export v7 (only worth keeping if the results above are good)

In [ ]:
import json as json_lib
export_dir_v7 = f'{PROJECT_DIR}/export_{RUN_NAME_V7}'
os.makedirs(export_dir_v7, exist_ok=True)
local_export = '/content/export_v7'
os.makedirs(local_export, exist_ok=True)
torch.save(model.state_dict(), f'{local_export}/wav2vec2_deepfake_classifier.pt')
with open(f'{local_export}/label_map.json', 'w') as f:
    json_lib.dump({'labels': LABELS, 'model_name': 'facebook/wav2vec2-base',
                   'note': 'v7 = binary genuine/deepfake, v2 fine-tuned on ASVspoof(VC)+DEEP-VOICE+'
                           'In-The-Wild(speaker split)+bank-fraud set, per-family class balance, '
                           'phone-quality augmentation; FakeAVCeleb held out'}, f, indent=2)
size = os.path.getsize(f'{local_export}/wav2vec2_deepfake_classifier.pt')
free = shutil.disk_usage(PROJECT_DIR).free
if size * 1.05 < free:
    shutil.copy2(f'{local_export}/wav2vec2_deepfake_classifier.pt', export_dir_v7)
    shutil.copy2(f'{local_export}/label_map.json', export_dir_v7)
    print('Exported to Drive:', export_dir_v7)
else:
    print(f'Drive too full for the export ({free/1e9:.2f} GB free) -- NOT saved to Drive.')

# Always also make a zip you can download to your laptop right now
shutil.make_archive('/content/export_v7', 'zip', local_export)
from google.colab import files
files.download('/content/export_v7.zip')
